In [1]:
!pip install pyspark

In [2]:
import urllib.request
import os

url = "https://www.gutenberg.org/cache/epub/100/pg100.txt"
archivo_base = "texto_base.txt"
archivo_dataset = "dataset_grande.txt"

print("Descargando texto base...")
urllib.request.urlretrieve(url, archivo_base)

print("Generando archivo de +100MB (esto tomará unos segundos)...")
with open(archivo_dataset, 'w', encoding='utf-8') as outfile:
    with open(archivo_base, 'r', encoding='utf-8') as infile:
        contenido = infile.read()
        for _ in range(22):
            outfile.write(contenido)

tamaño_mb = os.path.getsize(archivo_dataset) / (1024 * 1024)
print(f"¡Archivo generado exitosamente! Tamaño: {tamaño_mb:.2f} MB")

Descargando texto base...
Generando archivo de +100MB (esto tomará unos segundos)...
¡Archivo generado exitosamente! Tamaño: 114.18 MB


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("WordCount_RDD_vs_DataFrame") \
    .master("local[*]") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark Inicializado:", spark.version)

Spark Inicializado: 4.0.3


In [4]:
import time
import shutil

ruta_entrada = "dataset_grande.txt"
ruta_salida_rdd = "resultado_rdd"

shutil.rmtree(ruta_salida_rdd, ignore_errors=True)

print("Iniciando Word Count con RDD...")
tiempo_inicio_rdd = time.time()

rdd_lineas = sc.textFile(ruta_entrada)

rdd_conteo = rdd_lineas \
    .flatMap(lambda linea: linea.lower().split()) \
    .map(lambda palabra: (palabra, 1)) \
    .reduceByKey(lambda a, b: a + b)

rdd_conteo.saveAsTextFile(ruta_salida_rdd)

tiempo_fin_rdd = time.time()
tiempo_total_rdd = tiempo_fin_rdd - tiempo_inicio_rdd

print(f"Finalizado. Tiempo de ejecución RDD: {tiempo_total_rdd:.4f} segundos")

Iniciando Word Count con RDD...
Finalizado. Tiempo de ejecución RDD: 34.1111 segundos


In [5]:
from pyspark.sql.functions import split, explode, col, lower

ruta_salida_df = "resultado_df.parquet"
shutil.rmtree(ruta_salida_df, ignore_errors=True)

print("Iniciando Word Count con DataFrame...")
tiempo_inicio_df = time.time()

df_lineas = spark.read.text(ruta_entrada)

df_palabras = df_lineas.select(
    explode(split(lower(col("value")), "\\s+")).alias("palabra")
)

df_palabras = df_palabras.filter(col("palabra") != "")

df_conteo = df_palabras \
    .groupBy("palabra") \
    .count() \
    .orderBy(col("count").desc())

df_conteo.write.parquet(ruta_salida_df)

tiempo_fin_df = time.time()
tiempo_total_df = tiempo_fin_df - tiempo_inicio_df

print(f"Finalizado. Tiempo de ejecución DataFrame: {tiempo_total_df:.4f} segundos")

Iniciando Word Count con DataFrame...
Finalizado. Tiempo de ejecución DataFrame: 27.7132 segundos


In [6]:
if tiempo_total_rdd > tiempo_total_df:
    speedup = tiempo_total_rdd / tiempo_total_df
    mas_rapido = "DataFrame"
else:
    speedup = tiempo_total_df / tiempo_total_rdd
    mas_rapido = "RDD"

print("-" * 50)
print(" COMPARACIÓN DE RENDIMIENTO ".center(50, "="))
print("-" * 50)
print(f"Tiempo RDD:       {tiempo_total_rdd:.4f} segundos")
print(f"Tiempo DataFrame: {tiempo_total_df:.4f} segundos")
print("-" * 50)
print(f"El método con {mas_rapido} fue {speedup:.2f}x veces más rápido.")
print("-" * 50)

--------------------------------------------------
=========== COMPARACIÓN DE RENDIMIENTO ===========
--------------------------------------------------
Tiempo RDD:       34.1111 segundos
Tiempo DataFrame: 27.7132 segundos
--------------------------------------------------
El método con DataFrame fue 1.23x veces más rápido.
--------------------------------------------------


In [7]:
spark.stop()
print("Sesión de Spark finalizada.")

Sesión de Spark finalizada.
